# Simulation Time Scaling Analysis — Experiment 8

Wall-clock time analysis: how long each simulator takes to run as a function of workload parameters and NPU count.  
This notebook does **not** look at estimated execution time — only the real machine time consumed by G2, NS3, and Analytical.

In [28]:
import os
import re
import warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import plotly.io as pio
from typing import Dict

warnings.filterwarnings('ignore')
pio.renderers.default = "plotly_mimetype"

BASE_OUTPUT_DIR = '/app/astra-sim/upc/output/comparison_run/'
EXPERIMENT = 'experiment8'
TOPOLOGY = 'FoldedClosECMP1024'

PARAM_COLS = ['npu_count', 'd_model', 'num_stacks', 'seq_len', 'batch', 'micro_batch', 'dp', 'tp', 'pp', 'weight_sharded']
SIM_COLORS = {'G2': 'skyblue', 'NS3': 'lightgray', 'Analytical': 'salmon'}

print("Libraries loaded.")

Libraries loaded.


In [29]:
# --- Helper Functions ---

def parse_config(file_path: str) -> Dict[str, str]:
    """Parses a 'key: value' or 'key = value' configuration file."""
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = re.split(r'[:=]', line, 1)
                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except Exception:
        pass
    return params


def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into total seconds."""
    if not time_str:
        return 0.0
    try:
        parts = time_str.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    except (ValueError, IndexError):
        return 0.0


def parse_workload_params(workload_name: str) -> dict:
    """Parse model/parallelism params from workload directory name.
    Format: d{dmodel}_L{layers}_seq{seq}_b{batch}_mb{mb}_{dp}_{tp}_1_{pp}_{ws}
    """
    m = re.match(
        r'd(\d+)_L(\d+)_seq(\d+)_b(\d+)_mb(\d+)_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)',
        workload_name
    )
    if not m:
        return {}
    return {
        'd_model': int(m.group(1)),
        'num_stacks': int(m.group(2)),
        'seq_len': int(m.group(3)),
        'batch': int(m.group(4)),
        'micro_batch': int(m.group(5)),
        'dp': int(m.group(6)),
        'tp': int(m.group(7)),
        'sp': int(m.group(8)),
        'pp': int(m.group(9)),
        'weight_sharded': int(m.group(10)),
    }


# --- Data Collection ---

topo_base = os.path.join(BASE_OUTPUT_DIR, EXPERIMENT, TOPOLOGY)
raw_rows = []  # one row per (npu_count, workload, simulator, run)

if not os.path.isdir(topo_base):
    print(f"Output directory not found: {topo_base}")
else:
    for npu_dir in sorted(os.listdir(topo_base)):
        npu_path = os.path.join(topo_base, npu_dir)
        if not os.path.isdir(npu_path) or not npu_dir.startswith('npu_'):
            continue
        npu_count = int(npu_dir.split('_')[1])

        for workload_name in os.listdir(npu_path):
            workload_path = os.path.join(npu_path, workload_name)
            if not os.path.isdir(workload_path):
                continue

            wparams = parse_workload_params(workload_name)
            if not wparams:
                continue

            run_dirs = [d for d in os.listdir(workload_path) if d.startswith('run_')]
            for run_dir_name in run_dirs:
                run_path = os.path.join(workload_path, run_dir_name)
                sim_type = None
                for st in ['g2', 'ns3', 'analytical_unaware']:
                    if os.path.isdir(os.path.join(run_path, st)):
                        sim_type = st
                        break
                if not sim_type:
                    continue

                # Need at least a timing file to confirm the run completed
                sim_path = os.path.join(run_path, sim_type)
                has_output = any('trace_matched_timing.csv' in f for f in os.listdir(sim_path))
                if not has_output:
                    continue

                summary_params = parse_config(os.path.join(run_path, 'run_summary.txt'))
                sim_time_sec = parse_runtime(summary_params.get('total runtime', '0'))
                if sim_time_sec <= 0:
                    continue

                sim_name = {'g2': 'G2', 'ns3': 'NS3', 'analytical_unaware': 'Analytical'}[sim_type]
                raw_rows.append({
                    'workload': workload_name,
                    'simulator': sim_name,
                    'npu_count': npu_count,
                    'run': run_dir_name,
                    'sim_time_sec': sim_time_sec,
                    **wparams,
                })

print(f"Collected {len(raw_rows)} run records")


Collected 313 run records


In [30]:
# --- Build tidy DataFrame ---

if not raw_rows:
    raise SystemExit("No data found. Run simulations first.")

raw_df = pd.DataFrame(raw_rows)

# Aggregate runs per (simulator, npu_count, workload): mean, min, max
df = (
    raw_df
    .groupby(['simulator', 'npu_count', 'workload'] + [c for c in PARAM_COLS if c != 'npu_count'], as_index=False)
    .agg(
        sim_time_mean=('sim_time_sec', 'mean'),
        sim_time_min=('sim_time_sec', 'min'),
        sim_time_max=('sim_time_sec', 'max'),
        n_runs=('sim_time_sec', 'count'),
    )
    .rename(columns={'sim_time_mean': 'sim_time_sec'})
)

# Log-transformed columns (add small epsilon to avoid log(0))
eps = 1e-9
for col in ['sim_time_sec', 'npu_count', 'd_model', 'num_stacks', 'seq_len', 'batch', 'micro_batch']:
    df[f'log_{col}'] = np.log10(df[col].clip(lower=eps))

df.sort_values(['simulator', 'npu_count', 'workload'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Rows after aggregation: {len(df)}")
print(f"Simulators: {df['simulator'].unique()}")
print(f"NPU counts: {sorted(df['npu_count'].unique())}")
print(f"\nSim time range (sec):")
print(df.groupby('simulator')['sim_time_sec'].describe().round(2))


Rows after aggregation: 313
Simulators: ['Analytical' 'G2']
NPU counts: [2, 4, 8, 16, 32, 64, 128, 256]

Sim time range (sec):
            count   mean    std   min   25%   50%   75%     max
simulator                                                      
Analytical  159.0   5.57  14.25  0.49  0.67  1.22  4.09  126.15
G2          154.0  17.58  52.88  0.72  1.06  2.22  7.61  389.33


In [31]:
# --- Plot 1: Simulation Time vs NPU Count (log-log) with power-law fit ---

fig = go.Figure()

for sim in ['NS3', 'G2', 'Analytical']:
    sdf = df[df['simulator'] == sim].groupby('npu_count', as_index=False)['sim_time_sec'].mean()
    if sdf.empty or len(sdf) < 2:
        continue

    # Power-law fit: log10(t) = a * log10(npu) + b
    x_log = np.log10(sdf['npu_count'].values)
    y_log = np.log10(sdf['sim_time_sec'].values)
    coeffs = np.polyfit(x_log, y_log, 1)
    exponent, intercept = coeffs
    x_fit = np.linspace(x_log.min(), x_log.max(), 100)
    y_fit = np.polyval(coeffs, x_fit)

    color = SIM_COLORS[sim]
    # Scatter: actual measurements
    fig.add_trace(go.Scatter(
        x=sdf['npu_count'], y=sdf['sim_time_sec'],
        mode='markers', name=f'{sim}',
        marker=dict(size=10, color=color, line=dict(width=1, color='black')),
    ))
    # Fit line
    fig.add_trace(go.Scatter(
        x=10**x_fit, y=10**y_fit,
        mode='lines', name=f'{sim} fit (α={exponent:.2f})',
        line=dict(color=color, width=2, dash='dash'),
        showlegend=True,
    ))

fig.update_layout(
    title='Simulation Wall-Clock Time vs NPU Count (log-log)',
    xaxis_title='NPU Count', yaxis_title='Sim Time (s)',
    xaxis_type='log', yaxis_type='log',
    xaxis=dict(tickvals=[2, 4, 8, 16, 32, 64, 128]),
    template='plotly_white', font=dict(size=15),
    height=520, width=950,
    legend=dict(groupclick='toggleitem'),
)
fig.show()


In [32]:
# --- Plot 2: Sim Time vs Workload Parameters (d_model, num_stacks, seq_len, batch) ---
# 2x2 subplot grid, one panel per parameter, lines per simulator

param_axis_labels = {
    'd_model': 'Model Dimension (d_model)',
    'num_stacks': 'Number of Layers',
    'seq_len': 'Sequence Length',
    'batch': 'Batch Size',
}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=list(param_axis_labels.values()),
    shared_yaxes=False,
)

for idx, (param, label) in enumerate(param_axis_labels.items()):
    row, col = divmod(idx, 2)
    row += 1; col += 1

    for sim in ['NS3', 'G2', 'Analytical']:
        sdf = df[df['simulator'] == sim].groupby(param, as_index=False)['sim_time_sec'].mean()
        if sdf.empty:
            continue
        sdf = sdf.sort_values(param)
        fig.add_trace(
            go.Scatter(
                x=sdf[param], y=sdf['sim_time_sec'],
                mode='lines+markers', name=sim,
                marker=dict(size=8, color=SIM_COLORS[sim]),
                line=dict(color=SIM_COLORS[sim], width=2),
                showlegend=(idx == 0),
            ),
            row=row, col=col,
        )
    fig.update_xaxes(title_text=label, row=row, col=col)
    fig.update_yaxes(title_text='Sim Time (s)', row=row, col=col, type='log')

fig.update_layout(
    title='Mean Simulation Time vs Workload Parameters',
    template='plotly_white', font=dict(size=13),
    height=700, width=1050,
    legend=dict(title='Simulator'),
)
fig.show()


In [33]:
# --- Plot 3: Spearman Correlation Heatmap — params vs sim_time_sec ---

feature_cols = ['npu_count', 'd_model', 'num_stacks', 'seq_len', 'batch', 'micro_batch', 'dp', 'tp', 'pp']
feature_labels = ['NPU count', 'd_model', 'num_stacks', 'seq_len', 'batch', 'micro_batch', 'DP', 'TP', 'PP']

for sim in ['NS3', 'G2', 'Analytical']:
    sdf = df[df['simulator'] == sim].dropna(subset=feature_cols + ['sim_time_sec'])
    if len(sdf) < 5:
        continue

    corr_matrix = np.zeros((len(feature_cols), len(feature_cols) + 1))
    all_cols = feature_cols + ['sim_time_sec']
    all_labels = feature_labels + ['sim_time']

    # Full Spearman matrix among features + sim_time
    full_matrix = np.zeros((len(all_cols), len(all_cols)))
    for i, c1 in enumerate(all_cols):
        for j, c2 in enumerate(all_cols):
            r, _ = spearmanr(sdf[c1], sdf[c2])
            full_matrix[i, j] = r

    fig = go.Figure(go.Heatmap(
        z=full_matrix,
        x=all_labels, y=all_labels,
        colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
        text=np.round(full_matrix, 2),
        texttemplate='%{text}',
        colorbar=dict(title='Spearman ρ'),
    ))
    fig.update_layout(
        title=f'Spearman Correlation Matrix — {sim}',
        template='plotly_white', font=dict(size=12),
        height=550, width=700,
        xaxis=dict(tickangle=-40),
    )
    fig.show()


In [34]:
# --- Section 7: Multiple Log-Linear Regression ---
# Fit: log10(sim_time) ~ a*log10(npu) + b*log10(d_model) + c*log10(num_stacks) + d*log10(seq_len) + e*log10(batch)
# Coefficients are interpretable as scaling exponents (power-law).

log_features = ['log_npu_count', 'log_d_model', 'log_num_stacks', 'log_seq_len', 'log_batch']
feat_labels  = ['log(NPU)', 'log(d_model)', 'log(num_stacks)', 'log(seq_len)', 'log(batch)']

regression_results = {}

for sim in ['NS3', 'G2', 'Analytical']:
    sdf = df[df['simulator'] == sim].dropna(subset=log_features + ['log_sim_time_sec'])
    if len(sdf) < 5:
        continue

    X = sdf[log_features].values
    y = sdf['log_sim_time_sec'].values

    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)
    r2 = model.score(X, y)

    regression_results[sim] = {
        'model': model,
        'r2': r2,
        'coefs': dict(zip(feat_labels, model.coef_)),
        'intercept': model.intercept_,
    }

    # Predicted vs Actual scatter
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=y, y=y_pred, mode='markers',
        marker=dict(color=SIM_COLORS[sim], size=7, opacity=0.7),
        name='samples',
    ))
    min_val = min(y.min(), y_pred.min())
    max_val = max(y.max(), y_pred.max())
    fig.add_trace(go.Scatter(
        x=[min_val, max_val], y=[min_val, max_val],
        mode='lines', line=dict(color='black', dash='dot'), name='perfect fit',
    ))
    fig.update_layout(
        title=f'{sim}: Log-Linear Regression — Predicted vs Actual (R²={r2:.3f})',
        xaxis_title='Actual log₁₀(sim_time)', yaxis_title='Predicted log₁₀(sim_time)',
        template='plotly_white', font=dict(size=14),
        height=480, width=600,
    )
    fig.show()

    print(f"\n{'='*50}")
    print(f"[{sim}]  R² = {r2:.4f}")
    print(f"  Intercept (log₁₀ scale): {model.intercept_:.4f}")
    for feat, coef in zip(feat_labels, model.coef_):
        print(f"  {feat:>20s}  exponent = {coef:+.4f}")



[G2]  R² = 0.5788
  Intercept (log₁₀ scale): -0.6143
              log(NPU)  exponent = +0.8160
          log(d_model)  exponent = -0.0461
       log(num_stacks)  exponent = +0.1498
          log(seq_len)  exponent = -0.0539
            log(batch)  exponent = +0.2063



[Analytical]  R² = 0.5762
  Intercept (log₁₀ scale): -0.8162
              log(NPU)  exponent = +0.6316
          log(d_model)  exponent = -0.0512
       log(num_stacks)  exponent = +0.1924
          log(seq_len)  exponent = -0.0155
            log(batch)  exponent = +0.1929


In [35]:
# --- Section 9: Predicted Sim Time Surface (2D slices) ---
# Use the fitted log-linear models to generate prediction heatmaps over:
#   - npu_count  x  d_model  (other params at median)
#   - npu_count  x  seq_len  (other params at median)

feature_order = ['log_npu_count', 'log_d_model', 'log_num_stacks', 'log_seq_len', 'log_batch']
feature_names = ['npu_count', 'd_model', 'num_stacks', 'seq_len', 'batch']

npu_range    = np.logspace(np.log10(2), np.log10(128), 30)
dmodel_range = np.logspace(np.log10(128), np.log10(8192), 30)
seq_range    = np.logspace(np.log10(64), np.log10(4096), 30)

slices = [
    ('d_model',  'log_d_model',  dmodel_range),
    ('seq_len',  'log_seq_len',  seq_range),
]

for sim in ['NS3', 'G2', 'Analytical']:
    if sim not in regression_results:
        continue
    model = regression_results[sim]['model']
    sdf = df[df['simulator'] == sim].dropna(subset=feature_names)
    if sdf.empty:
        continue

    medians = {c: np.log10(sdf[c].median()) for c in feature_names}

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f'NPU × {s[0]}' for s in slices],
    )

    for col_idx, (param_name, log_col, param_range) in enumerate(slices, start=1):
        Z = np.zeros((len(npu_range), len(param_range)))
        for i, npu in enumerate(npu_range):
            for j, pval in enumerate(param_range):
                row_vals = [medians[f] for f in feature_names]
                row_vals[feature_names.index('npu_count')] = np.log10(npu)
                row_vals[feature_names.index(param_name)] = np.log10(pval)
                # model uses feature_order: log_npu, log_d_model, log_num_stacks, log_seq_len, log_batch
                feat_vec = np.array([[
                    np.log10(npu),
                    medians['d_model'] if param_name != 'd_model' else np.log10(pval),
                    medians['num_stacks'],
                    medians['seq_len']  if param_name != 'seq_len'  else np.log10(pval),
                    medians['batch'],
                ]])
                Z[i, j] = 10 ** model.predict(feat_vec)[0]

        fig.add_trace(
            go.Heatmap(
                x=param_range, y=npu_range, z=Z,
                colorscale='Viridis',
                colorbar=dict(title='Pred. sim_time (s)', x=0.45 if col_idx == 1 else 1.0, len=0.9),
                zsmooth='best',
            ),
            row=1, col=col_idx,
        )
        fig.update_xaxes(title_text=param_name, type='log', row=1, col=col_idx)
        fig.update_yaxes(title_text='NPU Count', type='log',
                         tickvals=[2, 4, 8, 16, 32, 64, 128], row=1, col=col_idx)

    fig.update_layout(
        title=f'{sim}: Predicted Sim Time Surface (2D slices at median params)',
        template='plotly_white', font=dict(size=13),
        height=500, width=1050,
    )
    fig.show()


In [36]:
# --- Section 10: LaTeX Summary Table ---

feat_labels_short = ['log(NPU)', 'log(d_model)', 'log(num_stacks)', 'log(seq_len)', 'log(batch)']

latex_rows = []
for sim in ['NS3', 'G2', 'Analytical']:
    if sim not in regression_results:
        continue
    res = regression_results[sim]
    row = {'Simulator': sim, 'R²': res['r2']}
    for fl, coef in zip(feat_labels_short, res['model'].coef_):
        row[f'α({fl})'] = coef

    sdf = df[df['simulator'] == sim]
    for npu in sorted(sdf['npu_count'].unique()):
        mean_t = sdf[sdf['npu_count'] == npu]['sim_time_sec'].mean()
        row[f'Mean t @ {npu} NPUs (s)'] = mean_t

    latex_rows.append(row)

latex_sum = pd.DataFrame(latex_rows)

print("=== Regression Coefficients and Mean Sim Times ===")
display(latex_sum.set_index('Simulator').style.format('{:.3f}').background_gradient(axis=None, cmap='RdYlGn'))

# LaTeX export: regression table
reg_cols = ['Simulator', 'R²'] + [f'α({fl})' for fl in feat_labels_short]
reg_df = latex_sum[reg_cols].copy()

latex_str = reg_df.to_latex(
    index=False,
    float_format='{:.3f}'.format,
    caption='Power-law scaling exponents from log-linear regression of wall-clock simulation time (Experiment 8).',
    label='tab:exp8_sim_time_regression',
    position='!htbp',
    column_format='l' + 'c' * (len(reg_cols) - 1),
    escape=True,
)
latex_str = (
    latex_str
    .replace('\\toprule', '\\hline')
    .replace('\\midrule', '\\hline')
    .replace('\\bottomrule', '\\hline')
)
print("\n=== LaTeX: Regression Table ===")
print(latex_str)

# LaTeX export: mean sim time per NPU tier
npu_cols = [c for c in latex_sum.columns if 'Mean t @' in c]
time_df = latex_sum[['Simulator'] + npu_cols].copy()

latex_str2 = time_df.to_latex(
    index=False,
    float_format='{:.1f}'.format,
    caption='Mean wall-clock simulation time (seconds) per NPU count tier (Experiment 8).',
    label='tab:exp8_sim_time_per_npu',
    position='!htbp',
    column_format='l' + 'c' * len(npu_cols),
    escape=True,
)
latex_str2 = (
    latex_str2
    .replace('\\toprule', '\\hline')
    .replace('\\midrule', '\\hline')
    .replace('\\bottomrule', '\\hline')
)
print("\n=== LaTeX: Mean Sim Time per NPU Tier ===")
print(latex_str2)


=== Regression Coefficients and Mean Sim Times ===


,R²,α(log(NPU)),α(log(d_model)),α(log(num_stacks)),α(log(seq_len)),α(log(batch)),Mean t @ 2 NPUs (s),Mean t @ 4 NPUs (s),Mean t @ 8 NPUs (s),Mean t @ 16 NPUs (s),Mean t @ 32 NPUs (s),Mean t @ 64 NPUs (s),Mean t @ 128 NPUs (s),Mean t @ 256 NPUs (s)
Simulator,,,,,,,,,,,,,,
G2,0.579,0.816,-0.046,0.150,-0.054,0.206,1.447,1.411,2.173,3.749,15.272,47.476,56.110,nan
Analytical,0.576,0.632,-0.051,0.192,-0.015,0.193,1.066,0.910,1.284,1.880,5.530,15.481,10.016,24.451



=== LaTeX: Regression Table ===
\begin{table}[!htbp]
\caption{Power-law scaling exponents from log-linear regression of wall-clock simulation time (Experiment 8).}
\label{tab:exp8_sim_time_regression}
\begin{tabular}{lcccccc}
\hline
Simulator & R² & α(log(NPU)) & α(log(d\_model)) & α(log(num\_stacks)) & α(log(seq\_len)) & α(log(batch)) \\
\hline
G2 & 0.579 & 0.816 & -0.046 & 0.150 & -0.054 & 0.206 \\
Analytical & 0.576 & 0.632 & -0.051 & 0.192 & -0.015 & 0.193 \\
\hline
\end{tabular}
\end{table}


=== LaTeX: Mean Sim Time per NPU Tier ===
\begin{table}[!htbp]
\caption{Mean wall-clock simulation time (seconds) per NPU count tier (Experiment 8).}
\label{tab:exp8_sim_time_per_npu}
\begin{tabular}{lcccccccc}
\hline
Simulator & Mean t @ 2 NPUs (s) & Mean t @ 4 NPUs (s) & Mean t @ 8 NPUs (s) & Mean t @ 16 NPUs (s) & Mean t @ 32 NPUs (s) & Mean t @ 64 NPUs (s) & Mean t @ 128 NPUs (s) & Mean t @ 256 NPUs (s) \\
\hline
G2 & 1.4 & 1.4 & 2.2 & 3.7 & 15.3 & 47.5 & 56.1 & NaN \\
Analytical & 1.1 &